In [1]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

checks = {
    "Feature Groups": "list_feature_groups",
    "Model Packages": "list_model_packages",
    "Pipelines": "list_pipelines",
    "Processing Jobs": "list_processing_jobs",
    "Training Jobs": "list_training_jobs",
    "Endpoints": "list_endpoints",
    "Monitoring Schedules": "list_monitoring_schedules",
}

for label, method in checks.items():
    print(f"\n=== {label} ===")
    try:
        result = getattr(sm, method)(MaxResults=10)

        for key, value in result.items():
            if isinstance(value, list):
                if value:
                    for item in value:
                        print(item)
                else:
                    print("No resources found")
                break

    except Exception as e:
        print("ACCESS ERROR:", str(e)[:250])


=== Feature Groups ===
No resources found

=== Model Packages ===
No resources found

=== Pipelines ===
No resources found

=== Processing Jobs ===
No resources found

=== Training Jobs ===
{'TrainingJobName': 'sagemaker-xgboost-2024-04-23-00-17-17-973', 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:945737411191:training-job/sagemaker-xgboost-2024-04-23-00-17-17-973', 'CreationTime': datetime.datetime(2024, 4, 23, 0, 17, 18, 134000, tzinfo=tzlocal()), 'TrainingEndTime': datetime.datetime(2024, 4, 23, 0, 20, 14, 868000, tzinfo=tzlocal()), 'LastModifiedTime': datetime.datetime(2024, 4, 23, 0, 20, 15, 210000, tzinfo=tzlocal()), 'TrainingJobStatus': 'Completed', 'SecondaryStatus': 'Completed'}

=== Endpoints ===
No resources found

=== Monitoring Schedules ===
No resources found


In [2]:
import boto3
import time
import pandas as pd

region = "us-east-1"
bucket = "medquad-ai-cleaned"
role_arn = "arn:aws:iam::945737411191:role/LabRole"

sm = boto3.client("sagemaker", region_name=region)

feature_group_name = "medquad-feature-group"

feature_definitions = [
    {"FeatureName": "record_id", "FeatureType": "String"},
    {"FeatureName": "question", "FeatureType": "String"},
    {"FeatureName": "source", "FeatureType": "String"},
    {"FeatureName": "focus_area", "FeatureType": "String"},
    {"FeatureName": "input_token_length", "FeatureType": "Integral"},
    {"FeatureName": "target_token_length", "FeatureType": "Integral"},
    {"FeatureName": "event_time", "FeatureType": "Fractional"},
]

try:
    response = sm.create_feature_group(
        FeatureGroupName=feature_group_name,
        RecordIdentifierFeatureName="record_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=feature_definitions,
        OnlineStoreConfig={"EnableOnlineStore": False},
        OfflineStoreConfig={
            "S3StorageConfig": {
                "S3Uri": f"s3://{bucket}/feature-store/"
            }
        },
        RoleArn=role_arn,
        Description="MedQuad AI FLAN-T5 feature engineering metadata"
    )

    print("Feature Group creation requested.")
    print("ARN:", response["FeatureGroupArn"])

except Exception as e:
    print("Creation failed:", str(e)[:500])

Feature Group creation requested.
ARN: arn:aws:sagemaker:us-east-1:945737411191:feature-group/medquad-feature-group


In [3]:
import boto3
import time

sm = boto3.client("sagemaker", region_name="us-east-1")

feature_group_name = "medquad-feature-group"

for attempt in range(12):
    response = sm.describe_feature_group(
        FeatureGroupName=feature_group_name
    )

    status = response["FeatureGroupStatus"]
    print(f"Attempt {attempt + 1}: {status}")

    if status == "Created":
        print("SUCCESS: Feature Group is ready.")
        print("Feature Group ARN:", response["FeatureGroupArn"])
        print("Offline Store:", response.get("OfflineStoreConfig"))
        break

    if status in ("CreateFailed", "DeleteFailed"):
        print("Failure reason:", response.get("FailureReason", "Not provided"))
        break

    time.sleep(5)
else:
    print("Feature Group is still being created. Check again shortly.")

Attempt 1: Created
SUCCESS: Feature Group is ready.
Feature Group ARN: arn:aws:sagemaker:us-east-1:945737411191:feature-group/medquad-feature-group
Offline Store: {'S3StorageConfig': {'S3Uri': 's3://medquad-ai-cleaned/feature-store/', 'ResolvedOutputS3Uri': 's3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data'}, 'DisableGlueTableCreation': False, 'DataCatalogConfig': {'TableName': 'medquad_feature_group_1792316115', 'Catalog': 'AwsDataCatalog', 'Database': 'sagemaker_featurestore'}}


In [6]:
import boto3
import pandas as pd
import time
from pathlib import Path
from transformers import AutoTokenizer

# 1. Load existing RAG passage dataset
csv_path = Path(
    "/home/sagemaker-user/medical_chatbot/data/processed/"
    "medquad_final_passages.csv"
)

df = pd.read_csv(csv_path)

# 2. Select 10 valid records
required = ["record_id", "question", "passage", "source", "focus_area"]
df = df.dropna(subset=["question", "passage"])

missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

sample = df.head(10).copy()

# 3. Calculate token-length features
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")

sample["input_token_length"] = sample["question"].apply(
    lambda q: len(
        tokenizer.encode(
            "answer medical question: " + str(q),
            add_special_tokens=True,
            truncation=False
        )
    )
)

sample["target_token_length"] = sample["passage"].apply(
    lambda p: len(
        tokenizer.encode(
            str(p),
            add_special_tokens=True,
            truncation=False
        )
    )
)

# 4. Connect to Feature Store
runtime = boto3.client(
    "sagemaker-featurestore-runtime",
    region_name="us-east-1"
)

event_time = str(time.time())

# 5. Insert records
for number, (_, row) in enumerate(sample.iterrows(), start=1):

    # Include chunk identifiers so separate passages
    # do not accidentally overwrite one another.
    record_id = (
        f"{row['record_id']}-"
        f"{row['chunk_id']}-"
        f"{row['subchunk_id']}"
    )

    record = [
        {"FeatureName": "record_id", "ValueAsString": record_id},
        {"FeatureName": "question", "ValueAsString": str(row["question"])},
        {"FeatureName": "source", "ValueAsString": str(row["source"])},
        {"FeatureName": "focus_area", "ValueAsString": str(row["focus_area"])},
        {
            "FeatureName": "input_token_length",
            "ValueAsString": str(int(row["input_token_length"]))
        },
        {
            "FeatureName": "target_token_length",
            "ValueAsString": str(int(row["target_token_length"]))
        },
        {"FeatureName": "event_time", "ValueAsString": event_time}
    ]

    runtime.put_record(
        FeatureGroupName="medquad-feature-group",
        Record=record
    )

    print(f"Inserted record {number}/10")

print("\nSUCCESS: 10 feature records submitted.")

Inserted record 1/10
Inserted record 2/10
Inserted record 3/10
Inserted record 4/10
Inserted record 5/10
Inserted record 6/10
Inserted record 7/10
Inserted record 8/10
Inserted record 9/10
Inserted record 10/10

SUCCESS: 10 feature records submitted.


In [7]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")
s3 = boto3.client("s3", region_name="us-east-1")

fg = sm.describe_feature_group(
    FeatureGroupName="medquad-feature-group"
)

s3_uri = fg["OfflineStoreConfig"]["S3StorageConfig"]["ResolvedOutputS3Uri"]

bucket, prefix = s3_uri.replace("s3://", "", 1).split("/", 1)

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=prefix,
    MaxKeys=100
)

objects = response.get("Contents", [])

print("Feature Group:", fg["FeatureGroupName"])
print("Status:", fg["FeatureGroupStatus"])
print("Offline Store:", s3_uri)
print("S3 objects found:", len(objects))

for obj in objects[:10]:
    print(obj["Key"], "|", obj["Size"], "bytes")

if any(obj["Key"].endswith(".parquet") for obj in objects):
    print("\nSUCCESS: Feature Store Parquet data found in S3.")
else:
    print("\nData not visible yet. Offline-store delivery may take several minutes.")

Feature Group: medquad-feature-group
Status: Created
Offline Store: s3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data
S3 objects found: 0

Data not visible yet. Offline-store delivery may take several minutes.


In [8]:
import boto3

s3 = boto3.client("s3", region_name="us-east-1")

bucket = "medquad-ai-cleaned"

extensions = (
    ".tar.gz",
    ".safetensors",
    ".bin",
    ".pt",
    ".pth",
    ".ckpt"
)

paginator = s3.get_paginator("list_objects_v2")

found = []

for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        if key.lower().endswith(extensions):
            found.append({
                "key": key,
                "size_mb": round(obj["Size"] / (1024 ** 2), 2)
            })

print("=== MODEL ARTIFACT SEARCH ===")
print("Artifacts found:", len(found))

for artifact in found[:30]:
    print(
        f"s3://{bucket}/{artifact['key']}",
        f"({artifact['size_mb']} MB)"
    )

if not found:
    print("No model weight artifacts found in this S3 bucket.")

=== MODEL ARTIFACT SEARCH ===
Artifacts found: 3
s3://medquad-ai-cleaned/data/processed/medgemma_evaluation/medgemma_source.tar.gz (0.0 MB)
s3://medquad-ai-cleaned/medgemma/experiments/mq019/code/medgemma_source.tar.gz (0.0 MB)
s3://medquad-ai-cleaned/medquad-mq019-20261017-121517/source/sourcedir.tar.gz (0.0 MB)


In [9]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

OUTPUT_DIR = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "artifacts/flan-t5-small-baseline"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading pretrained FLAN-T5 Small...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Saving model and tokenizer...")

model.save_pretrained(
    OUTPUT_DIR,
    safe_serialization=True
)

tokenizer.save_pretrained(OUTPUT_DIR)

print("\nSUCCESS: Pretrained baseline saved.")
print("Location:", OUTPUT_DIR)

for file in OUTPUT_DIR.iterdir():
    print(file.name, round(file.stat().st_size / 1024**2, 2), "MB")

Loading pretrained FLAN-T5 Small...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 776.33it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Saving model and tokenizer...


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.78s/it]


SUCCESS: Pretrained baseline saved.
Location: /home/sagemaker-user/medical_chatbot/artifacts/flan-t5-small-baseline
config.json 0.0 MB
generation_config.json 0.0 MB
model.safetensors 293.6 MB
tokenizer_config.json 0.0 MB
tokenizer.json 2.31 MB


In [10]:
import boto3
import tarfile
from pathlib import Path

model_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "artifacts/flan-t5-small-baseline"
)

archive_path = model_dir.parent / "flan-t5-small-baseline.tar.gz"

# Create SageMaker-compatible model archive
with tarfile.open(archive_path, "w:gz") as tar:
    for file in model_dir.iterdir():
        if file.is_file():
            tar.add(file, arcname=file.name)

print("Model archive created:", archive_path)
print(
    "Archive size:",
    round(archive_path.stat().st_size / (1024**2), 2),
    "MB"
)

# Upload to existing S3 bucket
bucket = "medquad-ai-cleaned"
key = "models/flan-t5-small-baseline/model.tar.gz"

s3 = boto3.client("s3", region_name="us-east-1")

s3.upload_file(
    str(archive_path),
    bucket,
    key
)

model_s3_uri = f"s3://{bucket}/{key}"

print("\nSUCCESS: Model uploaded to S3.")
print("Model artifact:", model_s3_uri)

Model archive created: /home/sagemaker-user/medical_chatbot/artifacts/flan-t5-small-baseline.tar.gz
Archive size: 272.93 MB

SUCCESS: Model uploaded to S3.
Model artifact: s3://medquad-ai-cleaned/models/flan-t5-small-baseline/model.tar.gz


In [11]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

group_name = "medquad-flan-t5-models"
model_s3_uri = (
    "s3://medquad-ai-cleaned/"
    "models/flan-t5-small-baseline/model.tar.gz"
)

# 1. Create Model Package Group if it does not exist
try:
    response = sm.create_model_package_group(
        ModelPackageGroupName=group_name,
        ModelPackageGroupDescription=(
            "MedQuad AI model registry for medical QA research"
        )
    )
    print("Model Package Group created.")

except sm.exceptions.ResourceInUse:
    print("Model Package Group already exists.")

# 2. Register pretrained FLAN-T5 baseline
response = sm.create_model_package(
    ModelPackageGroupName=group_name,
    ModelPackageDescription=(
        "Pretrained google/flan-t5-small baseline. "
        "Not fine-tuned on MedQuAD."
    ),
    InferenceSpecification={
        "Containers": [
            {
                "Image": (
                    "763104351884.dkr.ecr.us-east-1.amazonaws.com/"
                    "huggingface-pytorch-inference:2.1.0-transformers4.37.0-cpu-py310-ubuntu22.04"
                ),
                "ModelDataUrl": model_s3_uri,
                "Environment": {
                    "HF_TASK": "text2text-generation"
                }
            }
        ],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": [
            "ml.m5.large"
        ]
    },
    ModelApprovalStatus="PendingManualApproval"
)

print("\nSUCCESS: Model registration requested.")
print("Model Package ARN:", response["ModelPackageArn"])

Model Package Group created.

SUCCESS: Model registration requested.
Model Package ARN: arn:aws:sagemaker:us-east-1:945737411191:model-package/medquad-flan-t5-models/1


In [12]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")

model_arn = (
    "arn:aws:sagemaker:us-east-1:945737411191:"
    "model-package/medquad-flan-t5-models/1"
)

response = sm.describe_model_package(
    ModelPackageName=model_arn
)

print("=== MEDQUAD MODEL REGISTRY ===")
print("Model Package ARN:", response["ModelPackageArn"])
print("Package Status:", response["ModelPackageStatus"])
print("Approval Status:", response["ModelApprovalStatus"])
print("Version:", response["ModelPackageVersion"])

containers = response["InferenceSpecification"]["Containers"]

for container in containers:
    print("Model Artifact:", container.get("ModelDataUrl"))
    print("Container Image:", container.get("Image"))

if response["ModelPackageStatus"] == "Completed":
    print("\nSUCCESS: Model Registry verification completed.")
else:
    print("\nRegistration is not yet completed.")
    print("Failure reason:", response.get("FailureReason", "None reported"))

=== MEDQUAD MODEL REGISTRY ===
Model Package ARN: arn:aws:sagemaker:us-east-1:945737411191:model-package/medquad-flan-t5-models/1
Package Status: Completed
Approval Status: PendingManualApproval
Version: 1
Model Artifact: s3://medquad-ai-cleaned/models/flan-t5-small-baseline/model.tar.gz
Container Image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:2.1.0-transformers4.37.0-cpu-py310-ubuntu22.04

SUCCESS: Model Registry verification completed.


In [14]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

# Existing processed MedQuAD dataset
data_path = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/medquad_final_passages.csv"
)

df = pd.read_csv(data_path)

required_columns = [
    "record_id", "question", "passage",
    "source", "focus_area", "token_length"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Data quality metrics
report = {
    "report_type": "MedQuad Data Quality Monitoring",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "dataset": str(data_path),
    "total_records": int(len(df)),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_values": {
        col: int(df[col].isna().sum())
        for col in required_columns
    },
    "token_length": {
        "mean": float(df["token_length"].mean()),
        "median": float(df["token_length"].median()),
        "max": int(df["token_length"].max())
    },
    "source_distribution": {
        str(k): int(v)
        for k, v in df["source"].value_counts().items()
    },
    "checks": {
        "dataset_not_empty": len(df) > 0,
        "questions_present": bool(df["question"].notna().all()),
        "passages_present": bool(df["passage"].notna().all()),
        "token_lengths_nonnegative": bool(
            (df["token_length"].dropna() >= 0).all()
        )
    }
}

report["overall_status"] = (
    "PASS" if all(report["checks"].values()) else "FAIL"
)

# Save monitoring evidence
output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "monitoring/reports"
)
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "medquad_data_quality_report.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("=== MEDQUAD DATA QUALITY MONITORING ===")
print("Total records:", report["total_records"])
print("Duplicate rows:", report["duplicate_rows"])
print("Missing values:", report["missing_values"])
print("Token length metrics:", report["token_length"])
print("Overall status:", report["overall_status"])
print("Report saved:", output_file)

=== MEDQUAD DATA QUALITY MONITORING ===
Total records: 38544
Duplicate rows: 0
Missing values: {'record_id': 0, 'question': 0, 'passage': 0, 'source': 0, 'focus_area': 0, 'token_length': 0}
Token length metrics: {'mean': 149.20996782897467, 'median': 163.0, 'max': 240}
Overall status: PASS
Report saved: /home/sagemaker-user/medical_chatbot/monitoring/reports/medquad_data_quality_report.json


In [15]:
import boto3
from pathlib import Path

report_path = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "monitoring/reports/medquad_data_quality_report.json"
)

bucket = "medquad-ai-cleaned"
key = "monitoring/reports/medquad_data_quality_report.json"

s3 = boto3.client("s3", region_name="us-east-1")

s3.upload_file(
    str(report_path),
    bucket,
    key
)

# Verify the uploaded object exists
response = s3.head_object(
    Bucket=bucket,
    Key=key
)

print("=== MONITORING REPORT UPLOAD ===")
print("Status: SUCCESS")
print("S3 URI:", f"s3://{bucket}/{key}")
print("Object size:", response["ContentLength"], "bytes")
print("Last modified:", response["LastModified"])

=== MONITORING REPORT UPLOAD ===
Status: SUCCESS
S3 URI: s3://medquad-ai-cleaned/monitoring/reports/medquad_data_quality_report.json
Object size: 889 bytes
Last modified: 2026-10-18 10:00:35+00:00


In [16]:
import boto3

sm = boto3.client("sagemaker", region_name="us-east-1")
s3 = boto3.client("s3", region_name="us-east-1")

group = sm.describe_feature_group(
    FeatureGroupName="medquad-feature-group"
)

offline_uri = group["OfflineStoreConfig"]["S3StorageConfig"]["S3Uri"]

bucket = offline_uri.replace("s3://", "").split("/")[0]
prefix = offline_uri.replace(f"s3://{bucket}/", "").rstrip("/")

print("=== FEATURE STORE VERIFICATION ===")
print("Feature Group:", group["FeatureGroupName"])
print("Status:", group["FeatureGroupStatus"])
print("Offline Store:", offline_uri)

paginator = s3.get_paginator("list_objects_v2")

objects = []

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    objects.extend(page.get("Contents", []))

parquet_files = [
    obj for obj in objects
    if obj["Key"].endswith(".parquet")
]

print("Total S3 objects:", len(objects))
print("Parquet files:", len(parquet_files))

for obj in parquet_files[:5]:
    print(
        f"s3://{bucket}/{obj['Key']}",
        obj["Size"],
        "bytes"
    )

if parquet_files:
    print("\nSUCCESS: Offline Feature Store data exists.")
else:
    print("\nOffline records not yet visible at this prefix.")

=== FEATURE STORE VERIFICATION ===
Feature Group: medquad-feature-group
Status: Created
Offline Store: s3://medquad-ai-cleaned/feature-store/
Total S3 objects: 11
Parquet files: 10
s3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data/year=2026/month=10/day=18/hour=09/20261018T093822Z_0f889dd691293189.parquet 2517 bytes
s3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data/year=2026/month=10/day=18/hour=09/20261018T093822Z_312945429e508ccd.parquet 2437 bytes
s3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data/year=2026/month=10/day=18/hour=09/20261018T093822Z_6bc4d7bf6bf7e87f.parquet 2539 bytes
s3://medquad-ai-cleaned/feature-store/945737411191/sagemaker/us-east-1/offline-store/medquad-feature-group-1792316115/data/year=2026/month=10/day=18/hour=09/20261018T093822Z_85971fa6ca63e9b

In [17]:
import boto3
import pandas as pd
import torch
import json
from pathlib import Path
from datetime import datetime, timezone
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "artifacts/flan-t5-small-baseline"
)

data_path = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "data/processed/medquad_final_passages.csv"
)

output_dir = Path(
    "/home/sagemaker-user/medical_chatbot/"
    "inference/outputs"
)
output_dir.mkdir(parents=True, exist_ok=True)

print("Loading FLAN-T5 baseline on CPU...")

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)
model.to("cpu")
model.eval()

df = pd.read_csv(data_path)

# Fixed sample for reproducible demonstration
samples = df.dropna(subset=["question"]).head(5)

results = []

for _, row in samples.iterrows():
    question = str(row["question"])

    prompt = f"Answer this medical question briefly: {question}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    results.append({
        "record_id": str(row["record_id"]),
        "question": question,
        "model_prediction": prediction,
        "model": "google/flan-t5-small",
        "model_type": "pretrained_baseline",
        "generated_at": datetime.now(timezone.utc).isoformat()
    })

output_file = output_dir / "flan_t5_batch_predictions.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("\n=== BATCH INFERENCE RESULTS ===")

for item in results:
    print("\nQuestion:", item["question"])
    print("Prediction:", item["model_prediction"])

print("\nPredictions generated:", len(results))
print("Local output:", output_file)

# Upload batch inference evidence to S3
bucket = "medquad-ai-cleaned"
key = "inference/outputs/flan_t5_batch_predictions.json"

s3 = boto3.client("s3", region_name="us-east-1")

s3.upload_file(str(output_file), bucket, key)
s3.head_object(Bucket=bucket, Key=key)

print("\nSUCCESS: Batch predictions uploaded and verified.")
print("S3 URI:", f"s3://{bucket}/{key}")

Loading FLAN-T5 baseline on CPU...


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 707.79it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



=== BATCH INFERENCE RESULTS ===

Question: What is (are) Glaucoma ?
Prediction: Glaucoma (are) Glaucoma is a genus of fungi in the family Cerambycidae.

Question: What is (are) Glaucoma ?
Prediction: Glaucoma (are) Glaucoma is a genus of fungi in the family Cerambycidae.

Question: What is (are) Glaucoma ?
Prediction: Glaucoma (are) Glaucoma is a genus of fungi in the family Cerambycidae.

Question: What causes Glaucoma ?
Prediction: Glaucoma is a common cause of glaucoma.

Question: What causes Glaucoma ?
Prediction: Glaucoma is a common cause of glaucoma.

Predictions generated: 5
Local output: /home/sagemaker-user/medical_chatbot/inference/outputs/flan_t5_batch_predictions.json

SUCCESS: Batch predictions uploaded and verified.
S3 URI: s3://medquad-ai-cleaned/inference/outputs/flan_t5_batch_predictions.json


In [18]:
import json
import boto3
from pathlib import Path
from datetime import datetime, timezone

base = Path("/home/sagemaker-user/medical_chatbot")

predictions_path = (
    base / "inference/outputs/flan_t5_batch_predictions.json"
)

with open(predictions_path, encoding="utf-8") as f:
    predictions = json.load(f)

# Explicit review of the five observed outputs.
# These labels are specific to this demonstration.
review_labels = [
    "factually_incorrect",
    "factually_incorrect",
    "factually_incorrect",
    "circular_uninformative",
    "circular_uninformative"
]

if len(predictions) != len(review_labels):
    raise ValueError(
        "Prediction count differs from the manually reviewed sample."
    )

report = {
    "report_type": "Offline Medical Output Quality Review",
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model": "google/flan-t5-small",
    "model_type": "pretrained_baseline",
    "evaluation_scope": "Five demonstration outputs; repeated questions",
    "sample_size": len(predictions),
    "unique_questions": len({
        p["question"] for p in predictions
    }),
    "reviewed_outputs": [
        {
            "question": p["question"],
            "prediction": p["model_prediction"],
            "review_label": label
        }
        for p, label in zip(predictions, review_labels)
    ],
    "review_summary": {
        "factually_incorrect": review_labels.count(
            "factually_incorrect"
        ),
        "circular_uninformative": review_labels.count(
            "circular_uninformative"
        )
    },
    "clinical_readiness": "NOT_VALIDATED",
    "limitations": (
        "Small, non-independent sample. Manual qualitative "
        "review only; not a medical accuracy benchmark."
    )
}

output_path = (
    base / "monitoring/reports/"
    "medquad_model_output_quality_report.json"
)
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

bucket = "medquad-ai-cleaned"
key = "monitoring/reports/medquad_model_output_quality_report.json"

s3 = boto3.client("s3", region_name="us-east-1")
s3.upload_file(str(output_path), bucket, key)
s3.head_object(Bucket=bucket, Key=key)

print("=== MODEL OUTPUT QUALITY MONITORING ===")
print("Reviewed outputs:", report["sample_size"])
print("Unique questions:", report["unique_questions"])
print("Review summary:", report["review_summary"])
print("Clinical readiness:", report["clinical_readiness"])
print("S3 report:", f"s3://{bucket}/{key}")
print("SUCCESS: Monitoring report uploaded and verified.")

=== MODEL OUTPUT QUALITY MONITORING ===
Reviewed outputs: 5
Unique questions: 2
Review summary: {'factually_incorrect': 3, 'circular_uninformative': 2}
Clinical readiness: NOT_VALIDATED
S3 report: s3://medquad-ai-cleaned/monitoring/reports/medquad_model_output_quality_report.json
SUCCESS: Monitoring report uploaded and verified.


In [19]:
import boto3
import json

cloudwatch = boto3.client("cloudwatch", region_name="us-east-1")

dashboard_name = "MedQuad-AI-Infrastructure-Monitoring"

dashboard = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0,
            "width": 12, "height": 6,
            "properties": {
                "title": "S3 Bucket Storage Size",
                "region": "us-east-1",
                "metrics": [
                    [
                        "AWS/S3",
                        "BucketSizeBytes",
                        "BucketName", "medquad-ai-cleaned",
                        "StorageType", "StandardStorage"
                    ]
                ],
                "period": 86400,
                "stat": "Average",
                "view": "timeSeries"
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 0,
            "width": 12, "height": 6,
            "properties": {
                "title": "S3 Number of Objects",
                "region": "us-east-1",
                "metrics": [
                    [
                        "AWS/S3",
                        "NumberOfObjects",
                        "BucketName", "medquad-ai-cleaned",
                        "StorageType", "AllStorageTypes"
                    ]
                ],
                "period": 86400,
                "stat": "Average",
                "view": "timeSeries"
            }
        },
        {
            "type": "text",
            "x": 0, "y": 6,
            "width": 24, "height": 3,
            "properties": {
                "markdown": (
                    "# MedQuad AI Monitoring\n"
                    "Feature Store: medquad-feature-group\n\n"
                    "Model Registry: medquad-flan-t5-models\n\n"
                    "Data quality report: "
                    "s3://medquad-ai-cleaned/monitoring/reports/"
                    "medquad_data_quality_report.json\n\n"
                    "Model output review: "
                    "s3://medquad-ai-cleaned/monitoring/reports/"
                    "medquad_model_output_quality_report.json\n\n"
                    "**Note:** No SageMaker inference endpoint "
                    "or Model Monitor schedule is running."
                )
            }
        }
    ]
}

response = cloudwatch.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard)
)

print("=== CLOUDWATCH DASHBOARD ===")
print("Dashboard:", dashboard_name)
print("Validation messages:", response.get("DashboardValidationMessages", []))

if response.get("DashboardValidationMessages"):
    print("Review dashboard validation messages.")
else:
    print("SUCCESS: Dashboard definition accepted.")

=== CLOUDWATCH DASHBOARD ===
Dashboard: MedQuad-AI-Infrastructure-Monitoring
Validation messages: []
SUCCESS: Dashboard definition accepted.


In [20]:
import json
from pathlib import Path

base = Path("/home/sagemaker-user/medical_chatbot/notebooks")

for name in ["medgemma_inference.ipynb", "main.ipynb"]:
    path = base / name
    print(f"\n=== {name} ===")

    if not path.exists():
        print("File not found")
        continue

    notebook = json.loads(path.read_text(encoding="utf-8"))
    cells = notebook.get("cells", [])

    code_cells = [c for c in cells if c.get("cell_type") == "code"]
    executed = [
        c for c in code_cells
        if c.get("execution_count") is not None
    ]

    print("Total cells:", len(cells))
    print("Code cells:", len(code_cells))
    print("Executed code cells:", len(executed))

    for cell in cells[:5]:
        source = "".join(cell.get("source", []))
        print("\n", cell.get("cell_type"), ":", source[:180])


=== medgemma_inference.ipynb ===
Total cells: 46
Code cells: 46
Executed code cells: 45

 code : # MQ-019: MedGemma 4B IT — Repository Access Verification

from huggingface_hub import HfApi

MODEL_ID = "google/medgemma-4b-it"

api = HfApi()
files = api.list_repo_files(MODEL_ID

 code : # MQ-019: SageMaker hardware assessment

import os
import shutil
import torch

# CPU and RAM
print("CPU cores:", os.cpu_count())

with open("/proc/meminfo") as f:
    for line in f

 code : import boto3

session = boto3.Session()
print("AWS Region:", session.region_name)

 code : import boto3
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name="us-east-1")

try:
    response = sm.list_training_jobs(MaxResults=5)

    prin

 code : import boto3
import pandas as pd

sm = boto3.client("sagemaker", region_name="us-east-1")

response = sm.list_training_jobs(
    MaxResults=20,
    SortBy="CreationTime",
    SortO

=== main.ipynb ===
Total cells: 17
Code cells: 17
Executed 

In [21]:
import json
from pathlib import Path

base = Path("/home/sagemaker-user/medical_chatbot/notebooks")

for name in ["medgemma_inference.ipynb", "main.ipynb"]:
    path = base / name
    print(f"\n{'=' * 55}")
    print(name)
    print("=" * 55)

    nb = json.loads(path.read_text(encoding="utf-8"))
    cells = nb.get("cells", [])

    code_cells = [c for c in cells if c["cell_type"] == "code"]
    executed = [c for c in code_cells if c.get("execution_count") is not None]

    errors = []

    for index, cell in enumerate(cells):
        for output in cell.get("outputs", []):
            if output.get("output_type") == "error":
                errors.append({
                    "cell": index + 1,
                    "error": output.get("ename"),
                    "message": output.get("evalue", "")[:180]
                })

    print("Total cells:", len(cells))
    print("Code cells:", len(code_cells))
    print("Executed code cells:", len(executed))
    print("Recorded errors:", len(errors))

    print("\nSections / first-line descriptions:")
    for index, cell in enumerate(cells):
        source = "".join(cell.get("source", [])).strip()
        if source:
            first_line = source.splitlines()[0][:110]
            print(f"Cell {index + 1}: {first_line}")

    print("\nRecorded errors:")
    for error in errors[:10]:
        print(error)

    if not errors:
        print("No saved error outputs.")


medgemma_inference.ipynb
Total cells: 46
Code cells: 46
Executed code cells: 45
Recorded errors: 0

Sections / first-line descriptions:
Cell 1: # MQ-019: MedGemma 4B IT — Repository Access Verification
Cell 2: # MQ-019: SageMaker hardware assessment
Cell 3: import boto3
Cell 4: import boto3
Cell 5: import boto3
Cell 6: import boto3
Cell 7: import boto3
Cell 8: import boto3
Cell 9: import boto3
Cell 10: # MQ-019: MedGemma inference configuration
Cell 11: # MQ-019: Evidence-grounded medical RAG prompt
Cell 12: from pathlib import Path
Cell 13: test_question = "What causes Hanhart syndrome?"
Cell 14: # MQ-019: Prototype evidence filter for cause-related questions
Cell 15: # MQ-019: Build filtered, evidence-grounded prompt
Cell 16: # MQ-019: MedGemma inference function
Cell 17: from pathlib import Path
Cell 18: # MQ-019: Save baseline RAG prompt
Cell 19: import json
Cell 20: import boto3
Cell 21: from pathlib import Path
Cell 22: from pathlib import Path
Cell 23: import boto3
Cell 24: imp